In [0]:
"""

Configures Spark session authentication to ADLS Gen2 using an Azure AD
Service Principal (OAuth 2.0 client credentials flow), with secrets
retrieved securely from an Azure Key Vault-backed Databricks secret scope.

This avoids storing or referencing raw storage account keys anywhere
in code, notebooks, or configuration — a service principal scoped to
'Storage Blob Data Contributor' on a single storage account is used instead.

Usage:
    %run ./utils/adls_auth
    df = spark.read.parquet("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips/")
"""

SECRET_SCOPE = "kv-nyc-taxi-scope"
STORAGE_ACCOUNT = "stdatalakenyctaxi"

def configure_adls_oauth(spark, storage_account: str = STORAGE_ACCOUNT, secret_scope: str = SECRET_SCOPE) -> None:
    """
    Configure the active Spark session to authenticate to the given ADLS Gen2
    storage account using OAuth client-credentials flow.

    Args:
        spark: the active SparkSession.
        storage_account: name of the target ADLS Gen2 storage account.
        secret_scope: name of the Databricks secret scope backed by Key Vault.

    Raises:
        RuntimeError: if any required secret cannot be retrieved, so a
        misconfiguration fails loudly at setup time rather than later
        during a read/write call.
    """
    try:
        client_id = dbutils.secrets.get(scope=secret_scope, key="sp-client-id")
        client_secret = dbutils.secrets.get(scope=secret_scope, key="sp-client-secret")
        tenant_id = dbutils.secrets.get(scope=secret_scope, key="sp-tenant-id")
    except Exception as e:
        raise RuntimeError(
            f"Failed to retrieve one or more secrets from scope '{secret_scope}'. "
            f"Verify the Key Vault-backed secret scope is correctly configured. Original error: {e}"
        )

    endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"

    spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    )
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", endpoint)

    print(f"[adls_auth] OAuth configured for storage account '{storage_account}'.")


# Auto-run when notebook is %run-included
configure_adls_oauth(spark)